In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import qmc

1. Latin Hypercube Sampling

In [14]:
#this variable controls how big the sampling CSVs are
sample_size = 1000

df = pd.read_csv("https://raw.githubusercontent.com/The1OGShaggy/pyMAISE_VarianceReductionBiasMitigationFinal/refs/heads/main/chf_combined.csv")

data = df.select_dtypes(include=[np.number]).dropna()

X = data.values
n_samples, n_features = X.shape

lower_bounds = X.min(axis=0)
upper_bounds = X.max(axis=0)

n_lhs_samples = sample_size

sampler = qmc.LatinHypercube(d=n_features)
lhs_unit = sampler.random(n=n_lhs_samples)

lhs_scaled = qmc.scale(lhs_unit, lower_bounds, upper_bounds)

lhs_df = pd.DataFrame(lhs_scaled, columns=data.columns)
lhs_df.to_csv("lhs_samples.csv", index=False)

print(lhs_df.head())

      D (m)      L (m)       P (kPa)  G (kg m-2s-1)     Tin (C)    Xe (-)  \
0  0.015833   4.760861  14784.050575    4358.904510   95.739164  0.739869   
1  0.008092   6.887996  16708.234015    2971.523626  272.243251  0.642607   
2  0.006470   5.122742  11077.831229    5038.330489  358.177999 -0.071620   
3  0.010213  15.223595  10043.125040     813.894157  184.874679  0.427106   
4  0.014619   6.697655   5742.487881    4869.525086  155.705365  0.041573   

   CHF (kW m-2)  
0   9749.202220  
1   9187.413926  
2   9838.061371  
3   3858.558674  
4   4871.191649  


2. Random Sampling

In [15]:
random_unit = np.random.rand(sample_size, n_features)

random_scaled = lower_bounds + random_unit * (upper_bounds - lower_bounds)

random_df = pd.DataFrame(random_scaled, columns=data.columns)
random_df.to_csv("random_samples.csv", index=False)

print(random_df.head())

      D (m)      L (m)       P (kPa)  G (kg m-2s-1)     Tin (C)    Xe (-)  \
0  0.012764   8.231949   8197.509887    5914.922842   17.743528  0.535630   
1  0.016143   5.144843  20805.057800    1166.201742  315.620555  0.327599   
2  0.010607  13.716513  16524.140430     141.800446  285.067865  0.598033   
3  0.013470  11.893936  13540.418099    5601.909938  311.879426 -0.078456   
4  0.011611  14.257636  14541.403605    1282.195103   67.627587 -0.364848   

   CHF (kW m-2)  
0   8250.970882  
1    373.575771  
2   5047.425678  
3   5610.612178  
4   4927.075833  


3. Stratified Sampling

In [17]:
st_data = df.select_dtypes(include=[np.number]).dropna().copy()

strat_features = list(st_data.columns[:7])

if len(strat_features) == 0:
    raise ValueError("No numeric columns available for stratification.")

n_bins = 4
bin_cols = []

for col in strat_features:
    bin_col = f"{col}_bin"
    
    try:
        st_data[bin_col] = pd.qcut(
            st_data[col],
            q=n_bins,
            labels=False,
            duplicates="drop"
        )
        bin_cols.append(bin_col)

    except ValueError:
        print(f"Skipping column {col} (not enough variation for binning)")

if len(bin_cols) == 0:
    raise ValueError("No valid stratification bins were created.")

st_data["composite_stratum"] = st_data[bin_cols].astype(str).agg("_".join, axis=1)

n_samples = sample_size
n_strata = st_data["composite_stratum"].nunique()
samples_per_stratum = max(1, n_samples // n_strata)

sampled = (
    st_data.groupby("composite_stratum", group_keys=False)
        .apply(lambda g: g.sample(
            n=min(len(g), samples_per_stratum),
            random_state=42
        ))
)

drop_cols = bin_cols + ["composite_stratum"]
existing_cols = [c for c in drop_cols if c in sampled.columns]

sampled = sampled.drop(columns=existing_cols)

sampled.to_csv("multi_stratified_samples.csv", index=False)

print(sampled.head())
print("Total samples:", len(sampled))

         D (m)     L (m)      P (kPa)  G (kg m-2s-1)     Tin (C)    Xe (-)  \
1602  0.004873  0.398876   464.910449     498.092621  109.977763  0.389867   
215   0.005918  0.510446   312.712384     101.267873   45.407290  0.681819   
1074  0.007429  0.816081   134.872621     233.344034   32.149319  0.571635   
2260  0.005975  0.696060   104.139259     440.353789   31.574790  0.605496   
142   0.005888  0.991567  4237.854926     766.457415   48.956487  0.775089   

      CHF (kW m-2)  
1602   1529.440214  
215     573.647755  
1074    962.883544  
2260   1697.157479  
142    2591.405909  
Total samples: 1267
